# 5. 교차검증 및 하이퍼파라미터 조정

서울시 따릉이 일별 수요 예측. [요구사항](../../docs/ml/REQUIREMENTS.md).

In [1]:
from pathlib import Path
import sys, json, time, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))
from src.ml.features import FEATURES, load_daily, make_features
from src.ml.splits import split_data, time_cv
from src.ml.models import model_candidates
from src.ml.evaluation import metrics, compare, grouped_metrics
from src.ml.tuning import tune_models
from sklearn.base import clone
from sklearn.inspection import permutation_importance

plt.rcParams.update({"figure.figsize": (10, 4), "font.family": "Malgun Gothic", "axes.unicode_minus": False})
OUT = ROOT / "outputs/ml"
OUT.mkdir(parents=True, exist_ok=True)
LOCAL = ROOT / "data/processed"
LOCAL.mkdir(parents=True, exist_ok=True)
daily = load_daily(ROOT)
frame = make_features(daily)
train, validation, final_train, test = split_data(frame)


In [2]:
searches, cv_results, settings, selected = tune_models(train[FEATURES], train.rentals)
cv_results.to_csv(OUT / "05_cv_candidates.csv", index=False)
settings.to_csv(OUT / "05_cv_settings.csv", index=False)
folds = pd.DataFrame([{
    "fold": i + 1, "train_end": f"{train.iloc[fit].date.max():%Y-%m-%d}",
    "validation_start": f"{train.iloc[val].date.min():%Y-%m-%d}", "validation_end": f"{train.iloc[val].date.max():%Y-%m-%d}",
    "train_rows": len(fit), "validation_rows": len(val)
} for i, (fit, val) in enumerate(time_cv().split(train))])
folds.to_csv(OUT / "05_cv_folds.csv", index=False)
display(folds)
display(cv_results.round(3))
display(settings.round(3))


,fold,train_end,validation_start,validation_end,train_rows,validation_rows
0,1,2025-05-16,2025-05-18,2025-07-01,107,45
1,2,2025-06-30,2025-07-02,2025-08-15,152,45
2,3,2025-08-14,2025-08-16,2025-09-29,197,45


,model,params,fold_1_MAE,fold_2_MAE,fold_3_MAE,mean_MAE,std_MAE,mean_fit_seconds
0,ridge,{'ridge__alpha': 0.1},82925.344,23288.355,77604.756,61272.818,26946.759,0.003
1,ridge,{'ridge__alpha': 1},79382.168,23988.542,68953.368,57441.359,24034.811,0.002
2,ridge,{'ridge__alpha': 10},62052.959,26175.560,44065.795,44098.105,14646.904,0.002
3,ridge,{'ridge__alpha': 100},41206.082,24613.517,31830.666,32550.088,6792.961,0.002
4,random_forest,"{'randomforestregressor__max_depth': 3, 'rando...",39871.403,24805.897,28675.617,31117.639,6388.269,0.126
5,random_forest,"{'randomforestregressor__max_depth': 3, 'rando...",36874.341,24358.745,28427.593,29886.893,5212.626,0.124
6,random_forest,"{'randomforestregressor__max_depth': 6, 'rando...",43290.726,24374.367,30402.141,32689.078,7890.066,0.142
7,random_forest,"{'randomforestregressor__max_depth': 6, 'rando...",37801.019,24050.012,29718.066,30523.032,5642.607,0.139
8,random_forest,"{'randomforestregressor__max_depth': None, 'ra...",43445.635,24254.367,30995.614,32898.539,7949.509,0.136
9,random_forest,"{'randomforestregressor__max_depth': None, 'ra...",37801.019,24075.541,29867.618,30581.392,5626.087,0.128


,model,setting,mean_MAE,std_MAE
0,ridge,default,44098.105,14646.904
1,ridge,tuned,32550.088,6792.961
2,random_forest,default,32689.078,7890.066
3,random_forest,tuned,29886.893,5212.626


- 시간순 3개 fold에서 가장 낮은 평균 MAE는 random_forest의 29,887건임.
- 해당 설정의 fold 표준편차는 5,213건으로 평가 시점에 따른 변동이 존재함.
- fold마다 전처리를 다시 학습하고 하루 gap을 두어 미래 타깃 유입을 방지함.

In [3]:
tuned_validation = compare(validation, {name: search.predict(validation[FEATURES]) for name, search in searches.items()})
tuned_validation.to_csv(OUT / "05_validation.csv")
chosen = searches[selected]
selection = {"model": selected, "params": chosen.best_params_, "cv_MAE": -chosen.best_score_,
             "selection_rule": "minimum mean CV MAE on training period; validation is diagnostic only",
             "features": FEATURES, "random_state": 42, "prediction_time": "D-1 00:00; targets through D-2",
             "test_previously_seen_in_EDA": True}
(OUT / "05_selection.json").write_text(json.dumps(selection, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
joblib.dump(chosen.best_estimator_, LOCAL / "ml_selected_pipeline.joblib")
display(tuned_validation.round(4))
display(selection)


,MAE,RMSE,WAPE,R2
model,,,,
ridge,22246.2310,31411.7265,0.2391,0.3004
random_forest,28066.7416,36478.4806,0.3017,0.0565


{'model': 'random_forest',
 'params': {'randomforestregressor__max_depth': 3,
  'randomforestregressor__min_samples_leaf': 7},
 'cv_MAE': np.float64(29886.893129104257),
 'selection_rule': 'minimum mean CV MAE on training period; validation is diagnostic only',
 'features': ['lag_2',
  'lag_7',
  'lag_14',
  'rolling_7',
  'rolling_28',
  'dow_sin',
  'dow_cos',
  'year_sin',
  'year_cos',
  'weekend'],
 'random_state': 42,
 'prediction_time': 'D-1 00:00; targets through D-2',
 'test_previously_seen_in_EDA': True}

- 학습 구간의 평균 CV MAE를 기준으로 random_forest을 선정하고 설정을 고정함.
- 선정 모델의 고정 검증 MAE는 28,067건임.
- 검증 점수는 추가 진단이며 테스트 결과로 설정을 변경하지 않음.